# 데이터 전처리 자동화 — 반복 작업을 구조화한다

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🛠 반복을 구조화하다 — 함수형 변환과 파이프라인

### — 손으로 하던 정제를 한 줄 파이프라인으로 —

---

## 📋 학습 목표

이 자습서를 마치면 여러분은 다음을 할 수 있습니다.

1. `apply`·`map`·벡터화의 동작과 성능 차이를 설명하고, 상황에 맞게 골라 쓸 수 있습니다.
2. 범주형 인코딩(Label·Ordinal·One-Hot)의 차이와 적합한 상황을 구분할 수 있습니다.
3. 수치 스케일링(Min-Max·Standard·Robust)의 차이와 분포 특성에 맞는 선택 기준을 설명할 수 있습니다.
4. method chaining과 `.pipe()`로 가독성·재사용성을 갖춘 전처리 파이프라인을 설계할 수 있습니다.
5. 원본 CSV → 정제 → 인코딩/스케일링 → 저장으로 이어지는 단일 파이프라인을 작성할 수 있습니다.
6. 팀원이 맥락 설명 없이 읽어도 의도가 드러나는 전처리 코드를 작성할 수 있습니다.

> 💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해보세요. 노트북을 마친 뒤 다시 돌아와 비교하면 성장이 눈에 보일 거예요.

## 📚 목차

| Part | 내용 | 핵심 질문 |
| --- | --- | --- |
| Part 0 | 분석 상황과 학습 지도 | 손으로 하던 정제, 한 번에 묶으려면? |
| Part 1 | apply · map · 벡터화 | 같은 결과를 어떤 방법으로, 얼마나 빠르게? |
| Part 2 | 범주형 인코딩 | 글자 카테고리를 어떤 숫자로 바꿀까? |
| Part 3 | 수치 스케일링 | 단위가 다른 숫자를 어떻게 한 자에 놓을까? |
| Part 4 | method chaining | 한 줄이 다음 줄의 입력이 되게 하려면? |
| Part 5 | `.pipe()` 파이프라인 | 함수 단위로 단계를 조립하려면? |
| 종합 실습 | end-to-end 파이프라인 | CSV → 정제 → 저장을 한 함수로 묶을 수 있을까? |
| 정리 | 핵심 요약 · 다음 시간 | 이 파이프라인이 대용량에서도 버틸까? |

## 분석 상황과 학습 지도

# 0. 분석 상황과 학습 지도

## 지난 시간에는 무엇을 했나요?

지난 며칠 동안 우리는 더럽혀진 데이터를 **손으로** 고쳤습니다.

- **D+003** 결측치·이상치를 진단·처리하는 법
- **D+004** 여러 테이블을 키로 병합하고 그룹별로 집계하는 법
- **D+005** 문자열(`str` 액세서·정규표현식)과 날짜(`dt` 액세서·`resample`)를 쓸 수 있는 형태로 정제하는 법

각각의 기법은 익혔지만, 실제 현장에서 이 작업은 **반복**됩니다. 새 달이 되면 같은 정제를 또 해야 하고, 데이터셋이 바뀌면 같은 코드를 또 복사·붙여넣기 합니다.

## 오늘의 여정

오늘은 "지난 며칠 동안 손으로 하던 정제를 **한 덩어리 코드**로 묶는 날"입니다. 결과 값은 똑같지만, 같은 작업을 새 데이터에 다시 적용할 때 **한 줄**이면 끝나도록 만듭니다.

```text
[Part 1] apply / map / 벡터화   →  값 하나하나에 같은 변환을 거는 세 가지 방법
   ↓
[Part 2] 범주형 인코딩            →  글자 카테고리를 모델이 읽을 숫자로
   ↓
[Part 3] 수치 스케일링            →  단위가 다른 숫자를 한 자에 놓기
   ↓
[Part 4] method chaining          →  한 줄이 다음 줄의 입력이 되게
   ↓
[Part 5] .pipe() 파이프라인       →  함수 단위로 단계를 조립
   ↓
[종합 실습] CSV → 정제 → 저장 end-to-end 파이프라인
```

## 이 자습서 사용법

이 노트북은 **혼자서도 끝까지 학습할 수 있도록** 만들었습니다. 다음 네 박자로 진행하세요.

- 📖 **읽고** — 개념 설명을 천천히 읽습니다.
- 💻 **실행하고** — 코드 셀을 위에서부터 순서대로 실행합니다. (단축키: `Shift + Enter`)
- ✏️ **고쳐보고** — "스스로 해보자!" 칸에서 코드를 직접 바꿔봅니다.
- 🤔 **답해보고** — 체크포인트 질문에 스스로 답해봅니다. 틀려도 괜찮습니다.

> ⚠ **주의:** 코드 셀은 반드시 **위에서부터 순서대로** 실행해야 합니다. 중간 셀을 건너뛰면 "앞에서 만든 변수가 없다"는 오류가 날 수 있어요. 오류는 잘못이 아니라 디버깅의 단서입니다.

## 오늘의 무대: 모두마켓의 "월간 정제 작업"

모두마켓에 들어온 지 몇 주, 여러분은 매달 같은 정제 작업을 반복하고 있습니다. 새 달의 주문 데이터가 들어오면, 결측을 채우고, 이상치를 자르고, 문자열을 정리하고, 카테고리를 숫자로 바꾸고, 숫자 단위를 맞추고, 저장합니다.

> 동료가 한마디 합니다.
> "그 작업, **함수 하나**로 만들어 두면 다음 달엔 한 줄이면 끝나지 않을까요?"

맞습니다. 오늘은 **반복을 구조화**해서, 다음 달의 여러분이 한 줄로 부를 수 있는 파이프라인을 만듭니다.

> **🎯 오늘의 핵심**
> **"한 번 손으로 하던 정제는, 그다음부터 함수로 한다."** 분석가의 무기는 결과가 아니라 *재현 가능한 절차* 입니다.

아래 셀을 실행하면 이번 달 모두마켓 주문 데이터가 만들어집니다. 결측·이상치·표기 혼재가 적당히 섞여 있는, 실제와 비슷한 상태입니다.

> **읽는 법:** 1,500건의 주문 중 일부는 일부러 더럽힌 상태입니다. 결측(amount, customer_age)·이상치(나이 999, 수량 80)·표기 혼재(`app`/`APP`, ` 서울 `/`Seoul`, `VIP`)가 섞여 있습니다. 오늘은 이 데이터가 한 줄짜리 파이프라인을 통과해 "분석 가능한 표"가 되어 나오는 것을 목표로 합니다.

이제 도구를 하나씩 만나봅시다. 출발점은 "값 하나하나에 같은 변환을 거는 방법"입니다.

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [5]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


## apply · map · 벡터화 — 같은 결과, 다른 속도

# 1. apply · map · 벡터화 — 같은 결과, 다른 속도

지난 시간(D+005)에 우리는 문자열을 정제할 때 `df["channel"].str.lower()` 같은 코드를 썼습니다. "전체 열에 한꺼번에" 변환이 적용됐죠. 이렇게 **컬럼 전체에 동시에 변환을 거는 방법**을 데이터 분석에서는 세 가지 이름으로 부릅니다 — `apply`, `map`, 그리고 **벡터화(vectorization)**.

같은 결과를 내더라도 **속도와 가독성이 다릅니다**. 그래서 셋의 차이를 알면, 손가락이 가는 대로가 아니라 의도를 가지고 도구를 고를 수 있습니다.

> ❓ **이 파트에서 답할 질문:** 같은 값-단위 변환을 `apply`로 할까, `map`으로 할까, 아니면 벡터 연산으로 할까?

## 💡 쉽게 말하면 — 한 줄로 늘어선 값에 같은 동작을 거는 세 가지 방식

```text
값들:  [9900, 19900, 29900, 49900, ...]

(a) apply/map  →  값 하나씩 꺼내 함수에 넣고 돌려받기  (반복문 느낌, 느림)
(b) 벡터화      →  전체 배열에 한 번에 연산을 거기      (한 방, 빠름)
```

`apply`와 `map`은 **함수**를 받아 값마다 적용합니다. 벡터화는 함수 없이 **연산자**(`+ - * /`)나 **pandas/NumPy 내장 메서드**(`.str.lower()`, `.dt.year`, `np.where(...)` 등)로 컬럼 전체에 한 번에 계산을 겁니다.

> 💡 **개념 연결:** 벡터화는 D+002 NumPy 편(`arr * 1.1`)에서 만났던 그 개념입니다. pandas의 컬럼은 속이 NumPy 배열이라, 그때 배운 "전체에 동시에"가 그대로 통합니다.

## 자세히 알아보기

| 도구 | 무엇을 받나 | 어디에 쓰나 | 대표 형태 |
| --- | --- | --- | --- |
| `Series.map` | **dict** 또는 **1-입력 함수** | 한 값 → 다른 값으로 **치환** | `s.map({"M":1, "F":0})` |
| `Series.apply` | **1-입력 함수** | 값마다 **사용자 정의 함수** 적용 | `s.apply(lambda x: x*1.1)` |
| `DataFrame.apply` | **1-입력 함수** | 행 또는 열 단위 변환 | `df.apply(func, axis=1)` |
| 벡터화 | (함수 없이) 연산자·내장 메서드 | 컬럼 전체에 한 번에 | `s * 1.1`, `s.str.lower()` |

> 📌 **실무 포인트:** "가능하면 벡터화 → 안 되면 `map` → 그래도 안 되면 `apply`" 순서로 고릅니다. 벡터화가 가장 빠르고, `apply`는 가장 유연하지만 가장 느립니다.

> **읽는 법:** 셋 다 결과가 같습니다. 그래서 **결과만 보고는** 어느 도구를 썼는지 알 수 없죠. 차이는 다음 두 셀(가독성과 속도)에서 드러납니다.

> **읽는 법:** "이 값을 저 값으로 바꿔라"가 명확한 dict 치환은 **`map`이 가장 읽기 좋습니다**. 같은 일을 `apply`로 하면 `lambda x: dict.get(x)` 같은 군더더기가 붙습니다. 의도를 가장 잘 드러내는 도구를 고르는 것이 곧 가독성입니다.

> **읽는 법:** `apply`/`map`은 값을 하나씩 함수에 넣어 돌려받는 구조라 파이썬 반복문에 가깝습니다. 벡터화는 NumPy의 C 코드로 한 번에 처리되므로 자릿수가 다른 속도가 나옵니다. 데이터가 작을 때는 차이가 크지 않지만, **수십만~수백만 건**이 되면 이 차이가 분석 가능 시간 자체를 결정합니다.

## 데이터로 확인해 봅시다

- `df["channel"].str.lower().str.strip()`처럼 **pandas 내장 메서드**가 있는 경우엔 거의 항상 그것이 답입니다. 이미 벡터화돼 있어 빠릅니다.
- "VIP 등급은 1.5, GOLD는 1.2, …" 같은 **사전 치환**은 `map`이 가장 가독성 좋습니다.
- 여러 컬럼을 함께 보고 조건부로 새 값을 만들 때는 `df.apply(func, axis=1)`가 어울립니다. 단, 느립니다 — 가능하면 `np.where`·`np.select`로 벡터화하세요.

> 📌 **다른 산업에서는?** 마케팅에서는 `customer_segment` 코드(`A`, `B`, `C`)를 가중치로 바꿀 때 `map`이, 금융에서는 신용점수 구간 라벨링에 `np.where`/`np.select` 벡터화가, 헬스케어에서는 진단 코드를 카테고리 분류로 매핑할 때 `map`이 가장 자연스럽습니다.

> **읽는 법:** `np.select`는 "조건 리스트, 그에 대응하는 값 리스트, 그리고 기본값"을 받아 if-elif-else 분기를 **벡터화**합니다. 결과는 `apply`와 같지만 속도는 훨씬 빠릅니다. **읽기에 무리 없을 정도까지는** 벡터화로 작성하는 것이 좋은 습관입니다.

### 스스로 해보자! ✏️ (1)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `orders["membership"]`을 `{basic: 1, silver: 2, gold: 3, vip: 4}` 가중치로 바꾼 새 컬럼 `membership_score`를 만드세요. 가장 읽기 좋은 도구를 고르세요.
2. **(응용)** `orders["customer_age"]`에 1을 더해 `age_next_year` 컬럼을 만드세요. 같은 일을 (a) `apply`, (b) 벡터화 두 방식으로 작성해 결과가 같은지 확인하세요.
3. **(생각해보기)** "이 작업은 셋 중 무엇이 가장 자연스러운가?"를 결정하는 기준은 무엇일까요? (힌트: 치환인지, 한 값에 함수를 거는 것인지, 컬럼 전체 연산인지)

<details>
<summary>💡 힌트 (클릭)</summary>

```python
weight = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
orders["membership_score"] = orders["membership"].map(weight)   # dict 치환은 map이 자연스러움
print(orders[["membership", "membership_score"]].head())

orders["age_next_year_apply"] = orders["customer_age"].apply(lambda x: x + 1)
orders["age_next_year_vec"]   = orders["customer_age"] + 1
print((orders["age_next_year_apply"] == orders["age_next_year_vec"]).all())
```

여기서 `"VIP"`(대문자)는 `weight` 딕셔너리에 없으므로 결과에 `NaN`이 생깁니다. 표기 혼재를 정제하지 않으면 인코딩에서 그대로 누락된다는 신호입니다. (Part 2의 인코딩 이전에 정제가 왜 필요한지 보여주는 장면입니다.)

</details>

### ✅ 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. `apply`, `map`, 벡터화 중 일반적으로 가장 빠른 것은 무엇인가요?
2. dict로 "이 값을 저 값으로" 치환할 때 가장 자연스러운 도구는 무엇인가요?
3. `if-elif-else` 분기를 벡터화하려면 어떤 함수를 떠올릴 수 있나요?

> 💡 **다음 Part 예고:** 지금까지는 **값을 변환**하는 법을 봤습니다. 이제 **글자 카테고리**(예: `region`, `membership`)를 모델이 읽을 **숫자**로 바꿀 차례입니다. "어떤 숫자로 바꿔야 하는가"가 카테고리의 성격에 따라 달라집니다.

In [6]:
# 예제: 같은 결과를 세 가지 방법으로 — 'amount'에 부가세 10% 더하기
amounts = orders["amount"].dropna().head(8)
print("원본:", amounts.tolist())

# (a) apply: 사용자 정의 함수 적용
def add_vat(x):
    return x * 1.1

by_apply = amounts.apply(add_vat)

# (b) map: 함수도 받지만, dict로 치환할 때 더 자연스러움
by_map = amounts.map(add_vat)

# (c) 벡터화: 함수 없이 연산자 하나
by_vec = amounts * 1.1

print("apply 결과:", by_apply.round(0).tolist())
print("map   결과:", by_map.round(0).tolist())
print("벡터화 결과:", by_vec.round(0).tolist())

원본: [749700.0, 19900.0, 19900.0, 29900.0, 59700.0, 389700.0, 19900.0, 89900.0]
apply 결과: [824670.0, 21890.0, 21890.0, 32890.0, 65670.0, 428670.0, 21890.0, 98890.0]
map   결과: [824670.0, 21890.0, 21890.0, 32890.0, 65670.0, 428670.0, 21890.0, 98890.0]
벡터화 결과: [824670.0, 21890.0, 21890.0, 32890.0, 65670.0, 428670.0, 21890.0, 98890.0]


In [7]:
# 예제: 'map'이 가장 어울리는 자리 — dict 치환
gender_kr_to_num = {"M": 1, "F": 0}
sample_gender = pd.Series(["M", "F", "F", "M", "F"])

# map은 dict를 그대로 받습니다. 이게 가장 자연스러움.
print("map(dict):", sample_gender.map(gender_kr_to_num).tolist())

# 같은 일을 apply로 하려면 dict.get을 래핑해야 합니다(가독성 떨어짐).
print("apply(lambda):", sample_gender.apply(lambda x: gender_kr_to_num.get(x)).tolist())

map(dict): [1, 0, 0, 1, 0]
apply(lambda): [1, 0, 0, 1, 0]


In [8]:
# 예제: 속도 비교 — 10만 건에 '부가세 10% 더하기'
big = pd.Series(np.random.randint(1000, 100_000, 100_000).astype(float))

# (a) apply
start = time.time()
_ = big.apply(lambda x: x * 1.1)
t_apply = time.time() - start

# (b) map
start = time.time()
_ = big.map(lambda x: x * 1.1)
t_map = time.time() - start

# (c) 벡터화
start = time.time()
_ = big * 1.1
t_vec = time.time() - start

print(f"apply  : {t_apply*1000:.1f} ms")
print(f"map    : {t_map*1000:.1f} ms")
print(f"벡터화 : {t_vec*1000:.1f} ms")
print(f"→ 벡터화가 apply보다 약 {t_apply/max(t_vec, 1e-9):.0f}배 빠릅니다 (환경에 따라 다름)")

apply  : 25.2 ms
map    : 23.3 ms
벡터화 : 0.0 ms
→ 벡터화가 apply보다 약 25176048배 빠릅니다 (환경에 따라 다름)


In [9]:
# 예제: 모두마켓 데이터에 셋을 모두 적용 — '주문 등급' 만들기
# 규칙: amount >= 100000 → 'high', 30000 이상 → 'mid', 그 외 'low'

# (a) apply (행 단위 — 가독성은 좋지만 느림)
def classify(amount):
    if pd.isna(amount):
        return np.nan
    if amount >= 100_000:
        return "high"
    elif amount >= 30_000:
        return "mid"
    return "low"

orders["grade_apply"] = orders["amount"].apply(classify)

# (b) 벡터화 (np.select가 'if-elif-else'의 벡터화 버전)
orders["grade_vec"] = np.select(
    [orders["amount"] >= 100_000, orders["amount"] >= 30_000],
    ["high", "mid"],
    default="low"
)
# 결측은 결측으로 두기
orders.loc[orders["amount"].isna(), "grade_vec"] = np.nan

# 결과가 같은지 확인 (결측 처리도 함께)
print("두 방식 결과 동일?:", (orders["grade_apply"].fillna("X") == orders["grade_vec"].fillna("X")).all())
print(orders[["amount", "grade_apply", "grade_vec"]].head())

두 방식 결과 동일?: True
     amount grade_apply grade_vec
0       NaN         NaN       NaN
1  749700.0        high      high
2   19900.0         low       low
3       NaN         NaN       NaN
4   19900.0         low       low


In [ ]:
# 스스로 해보자! (1)
# 아래 주석(#)을 지우고 빈칸(___)을 채운 뒤 실행해보세요.

# weight = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
# orders["membership_score"] = orders["membership"].___(weight)   # 1) map / apply 중 자연스러운 것은?
# print(orders[["membership", "membership_score"]].head())

# orders["age_next_year_apply"] = orders["customer_age"].apply(lambda x: ___)
# orders["age_next_year_vec"]   = orders["customer_age"] ___ 1
# print((orders["age_next_year_apply"] == orders["age_next_year_vec"]).all())

## 범주형 인코딩 — 글자를 숫자로, 단 의미를 보존하며

# 2. 범주형 인코딩 — 글자를 숫자로, 단 의미를 보존하며

모두마켓 데이터에는 `region`(서울/경기/…), `membership`(basic/silver/gold/vip), `channel`(web/app) 같은 **글자 카테고리** 컬럼이 많습니다. 이런 컬럼은 그대로는 모델 학습에 쓸 수 없고, **숫자**로 바꿔야 합니다. 이때 어떤 숫자로 바꾸느냐에 따라 모델이 데이터를 *다르게 해석*합니다.

> ❓ **이 파트에서 답할 질문:** 카테고리 값을 숫자로 바꿀 때, 그 숫자에 **순서·크기**의 의미가 있어야 할까? 없어야 할까?

## 💡 쉽게 말하면 — 같은 카테고리, 세 가지 변환

```text
원본:  ["서울", "부산", "경기", "서울", "대구"]

(a) Label Encoding     →  0, 1, 2, 0, 3                ← 순서 없는데 숫자가 생김(주의!)
(b) Ordinal Encoding   →  basic=1, silver=2, gold=3   ← 진짜 순서가 있을 때만
(c) One-Hot Encoding   →  서울/부산/경기/대구 각 1열   ← 순서가 없을 때 안전
```

핵심은 "이 카테고리에 **순서**가 있는가?" 한 질문입니다.

- `region`(서울/부산/…)은 순서가 **없습니다** → One-Hot이 안전합니다.
- `membership`(basic < silver < gold < vip)은 순서가 **있습니다** → Ordinal이 자연스럽습니다.

`region`을 Label로 바꾸면 모델은 `대구(3) > 서울(0)`이라는 가짜 크기 비교를 학습할 수 있습니다. 그래서 **순서가 없는 카테고리에 Label은 위험**합니다.

## 자세히 알아보기

| 인코딩 | 어떻게 바꾸나 | 어울리는 상황 | 주의점 |
| --- | --- | --- | --- |
| **Label Encoding** | 각 카테고리에 정수 한 개씩(임의 순서) | 트리 계열 모델, **순서가 진짜 있을 때** | 선형/거리 모델에 쓰면 가짜 순서 학습 |
| **Ordinal Encoding** | 사용자가 정한 순서대로 정수 매핑 | 순서가 명확한 카테고리(등급, 학년) | 순서를 직접 지정해야 함 |
| **One-Hot Encoding** | 카테고리마다 0/1 컬럼 한 개씩 | **순서 없는** 카테고리 | 카디널리티(고윳값 수)가 크면 컬럼 폭발 |

> 💡 **카디널리티(cardinality)** : 그 컬럼에 등장하는 고유한 값의 종류 수. `region`은 5개라 작고, `product_id`는 수천 개라 큽니다. One-Hot은 카디널리티가 클수록 컬럼 수가 폭발합니다.

> 📌 **실무 포인트:** 인코딩 전에 반드시 **표기 정제**(공백 제거·대소문자 통일)를 먼저 하세요. 그렇지 않으면 ` 서울 `, `Seoul`, `서울`이 *서로 다른 카테고리*로 인코딩됩니다. (지난 Part 1의 스스로 해보자!에서 `"VIP"`가 NaN으로 빠졌던 그 문제입니다.)

> **읽는 법:** ` 서울 `, `Seoul`이 모두 `서울`로 통합됐고, `VIP`도 `vip`로 합쳐졌습니다. 인코딩은 항상 **정제 다음**입니다.

> **읽는 법:** `pd.factorize`는 등장 순서대로 0, 1, 2… 를 붙입니다. 빠르고 간단하지만, **`0 < 1`이 의미를 갖지 않습니다**. 트리 계열 모델은 이런 임의 정수도 잘 다루지만, 선형 회귀·k-NN처럼 거리·크기를 쓰는 모델에는 부적합합니다.

> **읽는 법:** Ordinal은 **사람이 순서를 직접 정해서** dict로 넘깁니다. 등급·학년처럼 의미상 순서가 명확할 때만 씁니다. (지난 Part 1에서 `map`이 가장 자연스러운 도구라고 했던 그 자리입니다.)

> **읽는 법:** 카테고리 하나당 0/1 컬럼이 한 개씩 생깁니다. 모델은 각 카테고리를 **독립된 신호**로 받아들이고, 가짜 순서 학습 위험이 사라집니다. 대신 컬럼 수가 늘어나므로, 카디널리티가 큰 컬럼(예: `product_id`)에는 직접 쓰지 않습니다.

> 💡 **더 알고 싶다면 (선택):** scikit-learn의 `OneHotEncoder`는 `handle_unknown="ignore"` 옵션으로 학습 시 보지 못한 카테고리가 들어와도 안전하게 처리합니다. 실서비스에서는 이 안전장치가 중요합니다.

> **읽는 법:** 카디널리티가 500인 컬럼을 One-Hot으로 펼치면 500개의 컬럼이 새로 생깁니다. 데이터 크기가 커지고, 학습이 느려지며, 대부분 0인 희소(sparse) 행렬이 됩니다. 이럴 때는 **Target Encoding**·**Frequency Encoding** 같은 대안을 검토합니다(더 알고 싶다면 참고).

## 데이터로 확인해 봅시다

| 컬럼 | 추천 인코딩 | 이유 |
| --- | --- | --- |
| `region` (5종) | One-Hot | 순서 없음, 카디널리티 작음 |
| `membership` (4종) | Ordinal | basic < silver < gold < vip 순서 있음 |
| `channel` (2종, web/app) | One-Hot 또는 Label(=0/1) | 어차피 2종이면 둘이 같음 |
| `category` (5종) | One-Hot | 순서 없음 |
| `product_id` (수백~수천) | Target/Frequency Encoding | 카디널리티 큼 (오늘 범위 밖) |

> 📌 **다른 산업에서는?** 마케팅은 캠페인 코드(`A`/`B`/`C`)는 One-Hot, 고객 등급은 Ordinal로 다룹니다. 금융은 신용등급(`AAA` > `AA` > `A` > …)에 Ordinal, 산업분류 코드는 카디널리티가 커서 Target Encoding을 자주 씁니다. 헬스케어는 ICD 진단코드처럼 카디널리티가 매우 큰 경우 임베딩으로 옮겨가기도 합니다.

### 스스로 해보자! ✏️ (2)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `channel_clean`을 One-Hot으로 변환해 `orders`에 합쳐보세요. (`pd.concat` 또는 `df.join`)
2. **(응용)** `category` 컬럼을 One-Hot으로 변환할 때, 다중공선성(multicollinearity) 방지를 위해 첫 번째 카테고리를 떨어뜨리려면 어떤 옵션을 쓸까요? (힌트: `drop_first`)
3. **(생각해보기)** `region`은 5개라 One-Hot이 안전하다고 했습니다. 만약 100개라면, One-Hot 이외에 어떤 선택지를 검토해야 할까요?

<details>
<summary>💡 힌트 (클릭)</summary>

```python
ch_oh = pd.get_dummies(orders["channel_clean"], prefix="ch", dtype=int)
orders = pd.concat([orders, ch_oh], axis=1)
print(orders.filter(like="ch_").head())

cat_oh = pd.get_dummies(orders["category"], prefix="cat", dtype=int, drop_first=True)
print(cat_oh.shape)
```

`drop_first=True`는 첫 번째 카테고리를 떨어뜨려 컬럼 수를 하나 줄입니다. 선형 모델에서 **다중공선성**(컬럼들이 서로 완벽히 예측 가능한 상태)을 피하기 위해 자주 씁니다. 트리 모델에는 큰 차이가 없습니다.

</details>

### ✅ 짚고 넘어가기

1. Label·Ordinal·One-Hot의 결정적 차이를 한 문장씩으로 말할 수 있나요?
2. 순서가 없는 카테고리에 Label Encoding을 쓰면 어떤 문제가 생기나요?
3. One-Hot을 쓰기 어려운 상황(컬럼 폭발)은 어떤 경우인가요?

> 💡 **다음 Part 예고:** 카테고리를 숫자로 바꿨으니 이제 **숫자끼리의 단위**가 문제가 됩니다. 나이(20~60)와 금액(9,900~249,900)은 자릿수가 다르고, 이대로 두면 거리 기반 모델이 "큰 숫자"만 봅니다. 다음 Part는 **스케일링** 입니다.

In [ ]:
# 예제 준비: 표기 정제부터 — 인코딩의 전제
orders["region_clean"] = orders["region"].str.strip().replace({"Seoul": "서울"})
orders["membership_clean"] = orders["membership"].str.lower()
orders["channel_clean"] = orders["channel"].str.strip().str.lower()

print("region 정제 후:", sorted(orders["region_clean"].unique()))
print("membership 정제 후:", sorted(orders["membership_clean"].unique()))
print("channel 정제 후:", sorted(orders["channel_clean"].unique()))

In [ ]:
# (a) Label Encoding — 카테고리에 정수 한 개씩 (임의 순서)
# 가장 간단한 방법: pd.factorize  (또는 sklearn.preprocessing.LabelEncoder)
region_codes, region_labels = pd.factorize(orders["region_clean"])
print("매핑(임의 순서):", dict(enumerate(region_labels)))
print("앞 10개:", region_codes[:10])

In [ ]:
# (b) Ordinal Encoding — '진짜 순서'가 있는 카테고리에
# membership: basic < silver < gold < vip
member_order = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
orders["membership_ord"] = orders["membership_clean"].map(member_order)

print(orders[["membership_clean", "membership_ord"]].head())
print("\n결측 발생 여부:", orders["membership_ord"].isna().sum(), "건")

In [ ]:
# (c) One-Hot Encoding — 순서 없는 카테고리에 가장 안전
# pandas의 pd.get_dummies가 가장 간편합니다.
one_hot = pd.get_dummies(orders["region_clean"], prefix="region", dtype=int)
print("One-Hot 결과(앞 5행):")
display(one_hot.head())
print("컬럼 수:", one_hot.shape[1], "— region 고윳값 수와 같습니다")

In [ ]:
# 예제: 카디널리티가 크면? — '상품 ID'를 One-Hot 하는 경우
fake_product = pd.Series([f"P{str(i).zfill(4)}" for i in np.random.randint(1, 500, 1500)])
print("고윳값 수:", fake_product.nunique())
print("One-Hot 시 만들어질 컬럼 수:", fake_product.nunique())
# 실제로는 만들지 않습니다 — 메모리가 폭발합니다.

In [ ]:
# 스스로 해보자! (2)
# 1) channel_clean One-Hot
# ch_oh = pd.get_dummies(orders["channel_clean"], prefix="ch", dtype=int)
# orders = pd.concat([orders, ch_oh], axis=1)
# print(orders.filter(like="ch_").head())

# 2) drop_first 옵션
# cat_oh = pd.get_dummies(orders["category"], prefix="cat", dtype=int, drop_first=___)
# print(cat_oh.shape)   # 컬럼이 한 개 줄어든 것을 확인

## 수치 스케일링 — 단위가 다른 숫자를 한 자에 놓기

# 3. 수치 스케일링 — 단위가 다른 숫자를 한 자에 놓기

`customer_age`(예: 35), `quantity`(예: 2), `amount`(예: 49,900)는 모두 숫자지만 **자릿수가 다릅니다**. 거리·유사도·크기를 비교하는 모델(k-NN, k-Means, 선형 회귀, SVM 등)은 *큰 숫자 컬럼이 다른 컬럼을 압도*합니다. 이를 막기 위해 **스케일링(scaling)** — 서로 다른 단위의 숫자를 한 자(scale)에 놓는 변환 — 을 합니다.

> ❓ **이 파트에서 답할 질문:** 분포의 모양과 이상치 유무에 따라, 어떤 스케일링이 가장 안전할까?

## 💡 쉽게 말하면 — 같은 데이터, 세 가지 자

```text
원본 amount:   [9,900,  19,900,  29,900,  49,900,  249,900]  (1만~25만, 단위 큼)

(a) Min-Max     →  [0,    0.04,   0.08,   0.17,   1.00 ]    (0~1로 압축)
(b) Standard    →  [-0.5, -0.4,   -0.3,   -0.1,   2.0  ]    (평균 0, 표준편차 1)
(c) Robust      →  [-0.5, 0,      0.5,    1.5,    11.0 ]    (중앙값 0, IQR로 나눔)
```

세 방법 모두 "단위를 맞추는" 것은 같지만, **이상치(249,900)에 대한 반응이 다릅니다**.

- **Min-Max**: 최댓값에 끌려갑니다. 이상치가 있으면 나머지가 한 구석에 뭉칩니다.
- **Standard**: 평균·표준편차를 쓰므로 이상치에 약합니다(평균이 끌려감).
- **Robust**: 중앙값·IQR을 써서 **이상치에 강합니다**.

## 자세히 알아보기

| 스케일러 | 공식 | 결과 범위 | 어울리는 상황 |
| --- | --- | --- | --- |
| **Min-Max** | $(x - \min) / (\max - \min)$ | $[0, 1]$ | 이상치가 거의 없고, 0~1 범위가 필요할 때 |
| **Standard** | $(x - \mu) / \sigma$ | 대략 $[-3, 3]$ | 분포가 대략 정규에 가까울 때 |
| **Robust** | $(x - \text{median}) / \text{IQR}$ | 분포 의존 | 이상치가 섞여 있을 때(가장 안전한 기본값) |

기호 풀이: $\mu$는 평균, $\sigma$는 표준편차, $\text{median}$은 중앙값, $\text{IQR}$은 사분위 범위(Q3 − Q1).

> 💡 **개념 연결:** 지난 시간(D+003)에 IQR로 이상치를 진단했죠. **Robust Scaler가 그 IQR을 그대로 분모에 씁니다.** 이상치를 *제거*하지 않아도 그 영향을 *줄이는* 방법입니다.

> 📌 **실무 포인트:** "잘 모르겠으면 **Standard**부터 시도, 이상치가 의심되면 **Robust**." 트리 계열 모델(Random Forest·XGBoost)은 **스케일링이 필요 없습니다** — 분기 기준이 *순서*에 의존하기 때문입니다.

> **읽는 법:** `minmax`는 0과 1 사이에, `standard`는 평균 0·표준편차 1 근방에 놓입니다. `robust`는 명확한 고정 범위가 없지만, 중앙값을 0으로 맞추고 IQR로 나눕니다. 통계량을 비교하면 셋의 성격 차이가 한눈에 보입니다.

> **읽는 법:** Min-Max는 이상치(가장 큰 값)에 끌려가 나머지가 왼쪽에 뭉칩니다. Standard는 평균이 끌려가 분포가 왼쪽으로 치우치고, Robust는 중앙값을 기준으로 잡아 본체가 가운데로 옵니다. **이상치가 의심되면 Robust가 가장 안전한 선택**이라는 게 그림으로 보입니다.

> **읽는 법:** **`fit`은 학습 데이터에서만** 합니다. test 데이터까지 합쳐 `fit`하면 모델이 미래(평가 시점) 정보를 미리 보는 **데이터 누수(data leakage)** 가 됩니다. 학습 데이터로 `fit`한 통계량을 test에 그대로 `transform`하는 것이 약속입니다.

> 💡 **더 알고 싶다면 (선택):** 분포가 매우 치우친 변수(`log`-normal 같은)는 **로그 변환**(`np.log1p`)을 먼저 한 뒤 스케일링하면 더 안정적입니다. 가격·소득·인구 같은 변수에서 자주 씁니다.

## 데이터로 확인해 봅시다

- **스케일링이 필요한 모델:** k-NN, k-Means, SVM, 선형 회귀(정규화 포함), 로지스틱 회귀, 신경망.
- **스케일링이 필요 없는 모델:** Decision Tree, Random Forest, XGBoost, LightGBM. (순서만 보기 때문)
- 분석가는 모델을 정하기 전부터 **분포의 단위가 다르면 일단 스케일링을 의식**해 두는 게 좋습니다.

> 📌 **다른 산업에서는?** 마케팅은 광고비(만 원~억 원)와 클릭률(0~1) 같이 자릿수가 극단적으로 다른 변수를 함께 다룰 때 스케일링이 필수입니다. 금융은 신용 점수·소득·부채 같이 단위가 다른 변수를 거리 기반 사기탐지에 쓸 때, 헬스케어는 키·몸무게·혈압·검사 수치를 함께 입력하는 임상 모델에서 표준화가 기본입니다.

### 스스로 해보자! ✏️ (3)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** `quantity` 컬럼에 Min-Max 스케일러를 적용한 결과를 새 컬럼 `quantity_mm`으로 만드세요.
2. **(응용)** `quantity`에는 이상치(80)가 있습니다. Min-Max 결과의 분포를 히스토그램으로 그려보고, 이상치 때문에 본체가 한쪽에 뭉치는지 확인하세요.
3. **(생각해보기)** 같은 `quantity`에 Robust 스케일러를 적용하면 어떻게 달라질까요? 직접 비교해보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
qty = orders[["quantity"]]
orders["quantity_mm"]     = MinMaxScaler().fit_transform(qty)
orders["quantity_robust"] = RobustScaler().fit_transform(qty)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
sns.histplot(orders["quantity_mm"],     bins=30, ax=axes[0]); axes[0].set_title("Min-Max")
sns.histplot(orders["quantity_robust"], bins=30, ax=axes[1]); axes[1].set_title("Robust")
plt.tight_layout(); plt.show()
```

Min-Max는 이상치 80 때문에 정상 주문 1~3이 0 근처에 다 뭉칩니다. Robust는 IQR이 분모라 이상치의 영향이 훨씬 작아 본체가 잘 펴집니다.

</details>

### ✅ 짚고 넘어가기

1. Min-Max·Standard·Robust의 분모(나눠지는 양)는 각각 무엇인가요?
2. 이상치가 의심될 때 가장 안전한 기본값은 무엇인가요?
3. `fit`을 학습 데이터에만 적용해야 하는 이유는 무엇인가요?

> 💡 **다음 Part 예고:** 우리는 지금까지 한 단계씩 따로 했습니다 — 정제 → 인코딩 → 스케일링. 그런데 코드를 줄별로 보면 `df_step1 = ...`, `df_step2 = ...` 같이 **중간 변수**가 쌓입니다. 다음 Part는 이 단계들을 **한 줄로 흐르게** 묶는 **method chaining**입니다.

In [ ]:
# 예제: 세 가지 스케일러를 직접 비교 — 'amount'에 적용
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

amt = orders[["amount"]].dropna()   # 2D로 두는 게 sklearn 관례

mm = MinMaxScaler().fit_transform(amt)
ss = StandardScaler().fit_transform(amt)
rs = RobustScaler().fit_transform(amt)

scaled = pd.DataFrame({
    "original": amt["amount"].values,
    "minmax":   mm.flatten(),
    "standard": ss.flatten(),
    "robust":   rs.flatten(),
})
print(scaled.describe().round(2))

In [ ]:
# 예제: 그림으로 비교 — 이상치가 있을 때 각 스케일러가 어떻게 반응하나
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
sns.histplot(scaled["original"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].set_title("Original")

sns.histplot(scaled["minmax"], bins=40, ax=axes[1], color="#55A868")
axes[1].set_title("Min-Max [0, 1]")

sns.histplot(scaled["standard"], bins=40, ax=axes[2], color="#C44E52")
axes[2].set_title("Standard (mean=0)")

sns.histplot(scaled["robust"], bins=40, ax=axes[3], color="#8172B2")
axes[3].set_title("Robust (median=0)")

for ax in axes:
    ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# 예제: fit / transform 분리 — 실서비스의 정석
# 학습 데이터로 'fit'한 통계량을 새 데이터에도 똑같이 'transform'에 써야 합니다.
train = orders.iloc[:1000][["amount"]].dropna()
test  = orders.iloc[1000:][["amount"]].dropna()

scaler = StandardScaler()
scaler.fit(train)                              # train의 평균·표준편차를 기억
train_scaled = scaler.transform(train)         # 같은 기준으로 변환
test_scaled  = scaler.transform(test)          # test에도 같은 기준 적용

print("train 평균:", round(train_scaled.mean(), 3), "표준편차:", round(train_scaled.std(), 3))
print("test  평균:", round(test_scaled.mean(), 3),  "표준편차:", round(test_scaled.std(), 3))

In [ ]:
# 스스로 해보자! (3)
# from sklearn.preprocessing import MinMaxScaler, RobustScaler

# qty = orders[["quantity"]]
# orders["quantity_mm"]     = MinMaxScaler().fit_transform(qty)
# orders["quantity_robust"] = RobustScaler().fit_transform(qty)

# fig, axes = plt.subplots(1, 2, figsize=(12, 3))
# sns.histplot(orders["quantity_mm"],     bins=30, ax=axes[0]); axes[0].set_title("Min-Max")
# sns.histplot(orders["quantity_robust"], bins=30, ax=axes[1]); axes[1].set_title("Robust")
# plt.tight_layout(); plt.show()

## method chaining — 한 줄이 다음 줄의 입력이 되게

# 4. method chaining — 한 줄이 다음 줄의 입력이 되게

지금까지 우리는 데이터 정제를 다음처럼 단계별로 적었습니다.

```python
step1 = orders.dropna(subset=["amount"])
step2 = step1[step1["customer_age"] > 0]
step3 = step2.assign(amount_log=np.log1p(step2["amount"]))
result = step3.sort_values("amount_log")
```

코드가 길어질수록 `step1`, `step2`, … 같은 **임시 변수**가 쌓입니다. 나중에 보면 어떤 step이 무엇을 하는지 한눈에 안 들어옵니다.

> ❓ **이 파트에서 답할 질문:** 임시 변수를 만들지 않고, 단계의 흐름을 **한 줄로 읽히게** 적으려면?

## 💡 쉽게 말하면 — 컨베이어 벨트

pandas의 메서드(`dropna`, `query`, `assign`, `sort_values` 등)는 **거의 모두 새 DataFrame을 돌려줍니다**. 그래서 `.`을 점점 이어 붙이면 컨베이어 벨트처럼 흐릅니다.

```text
orders
  .dropna(subset=["amount"])           ← 1단계 가공
  .query("customer_age > 0")           ← 2단계 가공 (앞 결과를 입력으로)
  .assign(amount_log=lambda d: np.log1p(d["amount"]))  ← 3단계
  .sort_values("amount_log")           ← 4단계
```

이 패턴을 **method chaining**이라고 부릅니다. R의 `magrittr` 파이프(`%>%`), Unix의 `|`(파이프)와 같은 사고방식입니다.

## 자세히 알아보기

자주 쓰는 "체이닝 친화적" 메서드 모음:

| 메서드 | 하는 일 | 비고 |
| --- | --- | --- |
| `dropna(subset=[...])` | 특정 열의 결측 행 제거 | |
| `fillna({col: val})` | 결측 채우기 | |
| `query("조건식")` | 문자열 조건으로 필터 | 가독성이 매우 좋음 |
| `assign(new=...)` | 새 컬럼 추가 | `lambda d: d["col"]*2`로 직전 단계 참조 |
| `rename(columns={...})` | 컬럼명 바꾸기 | |
| `astype({col: dtype})` | 자료형 변환 | |
| `sort_values(by=...)` | 정렬 | |
| `reset_index(drop=True)` | 인덱스 리셋 | |
| `drop_duplicates()` | 중복 행 제거 | |

> 💡 **assign + lambda 패턴:** 새 컬럼을 만들 때 직전 단계의 DataFrame을 참조하려면 `assign(new=lambda d: d["x"] * 2)`처럼 **람다(lambda)** 를 씁니다. `d`는 직전 단계의 결과예요. 이게 체이닝의 핵심 트릭입니다.

> **읽는 법:** 두 방식은 결과가 완전히 같지만, **읽기 경험이 다릅니다**. (b)는 위에서 아래로 "이 데이터에 → 결측 제거 → 나이 범위 필터 → 새 컬럼 추가 → 정렬"이라는 **하나의 문장**처럼 읽힙니다. 임시 변수 `step1`, `step2`가 사라져 의도가 더 잘 드러납니다.

## query의 진가

`query("0 < customer_age < 120")` 같은 한국어에 가까운 표현은 pandas의 가장 강력한 가독성 무기 중 하나입니다.

```python
df[(df["age"] > 0) & (df["age"] < 120)]        # 괄호 지옥
df.query("0 < age < 120")                      # 자연스러움
```

조건이 길어질수록 차이는 더 크게 벌어집니다.

> **읽는 법:** `and`, `or`, `not`을 그대로 쓸 수 있고 **변수명을 따옴표 없이** 쓸 수 있어 가독성이 좋습니다. 백틱(`)으로 컬럼명을 감싸면 공백·특수문자가 있는 컬럼도 다룰 수 있습니다.

## 데이터로 확인해 봅시다

- 탐색적 정제(EDA 단계)에서는 단계 변수 방식이 디버깅에 좋습니다(중간 결과를 그때그때 볼 수 있으니).
- **확정된 정제 코드**는 method chaining으로 정리하면 다음 사람이 한 번에 읽고 이해할 수 있습니다.

> 📌 **다른 산업에서는?** 어디서나 같습니다. 마케팅 일일 리포트의 `df`→집계→정렬→상위 N 추출, 금융의 거래→필터→집계→리스크 점수 계산, 제조의 센서→이상치 제거→롤링 평균→경보 플래그까지, *흐름이 있는 작업은 체이닝과 잘 맞습니다*.

### 스스로 해보자! ✏️ (4)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

다음 정제를 **method chaining 한 번**으로 작성해보세요.

- `amount` 결측 제거
- `customer_age`가 0~120 사이인 행만 남기기
- `channel`을 소문자·공백제거(`channel_clean`)
- `amount_log = log1p(amount)` 컬럼 추가
- `amount`가 큰 순으로 정렬

> ⚠ **주의:** 한 번에 다 적고 한 번에 실행했을 때 결과가 깔끔히 나오면 성공입니다.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
result = (
    orders
    .dropna(subset=["amount"])
    .query("0 < customer_age < 120")
    .assign(
        channel_clean=lambda d: d["channel"].str.strip().str.lower(),
        amount_log=lambda d: np.log1p(d["amount"]),
    )
    .sort_values("amount", ascending=False)
    .reset_index(drop=True)
)
print(result.shape)
result.head(3)
```

`assign(... = lambda d: ...)`에서 `d`는 직전 단계까지 처리된 DataFrame입니다. 그래서 새 컬럼이 *체이닝 중간 상태*를 기준으로 만들어집니다. 이 한 줄 트릭이 chaining의 강점이에요.

</details>

### ✅ 짚고 넘어가기

1. method chaining의 가장 큰 장점을 한 문장으로 말할 수 있나요?
2. 체이닝 중에 새 컬럼을 만들 때 `assign(... = lambda d: ...)`처럼 람다를 쓰는 이유는 무엇인가요?
3. `df[df["age"] > 0]`와 `df.query("age > 0")` 중 어떤 것이 더 길어졌을 때 읽기 좋을까요?

> 💡 **다음 Part 예고:** Chaining이 흐름을 보기 좋게 만들었지만, **다른 데이터셋에 똑같이 적용**하려면 또 복사·붙여넣기를 해야 합니다. 다음 Part는 단계 자체를 **함수**로 만들어, 체이닝 안에서 함수 단위로 조립할 수 있게 해주는 **`.pipe()`** 입니다.

In [ ]:
# 예제: '단계 변수'로 쓴 정제와 'chaining'으로 쓴 정제 — 결과는 같음

# (a) 단계 변수 방식 (전통)
step1 = orders.dropna(subset=["amount", "customer_age"])
step2 = step1[(step1["customer_age"] > 0) & (step1["customer_age"] < 120)]
step3 = step2.assign(
    region_clean=step2["region"].str.strip().replace({"Seoul": "서울"}),
    amount_log=np.log1p(step2["amount"]),
)
clean_a = step3.sort_values("amount", ascending=False).reset_index(drop=True)

# (b) method chaining 방식 (체이닝)
clean_b = (
    orders
    .dropna(subset=["amount", "customer_age"])
    .query("0 < customer_age < 120")
    .assign(
        region_clean=lambda d: d["region"].str.strip().replace({"Seoul": "서울"}),
        amount_log=lambda d: np.log1p(d["amount"]),
    )
    .sort_values("amount", ascending=False)
    .reset_index(drop=True)
)

# 두 결과가 같은가?
print("두 방식 결과 동일?:", clean_a.equals(clean_b))
print("정제 후 행 수:", clean_b.shape[0])
clean_b.head(3)

In [ ]:
# 예제: query로 같은 필터 작성
# 1) 전통 방식
cond = (orders["customer_age"] > 0) & (orders["customer_age"] < 120) & (orders["amount"] >= 30000)
trad = orders[cond]

# 2) query 방식
q = orders.query("0 < customer_age < 120 and amount >= 30000")

print("동일?", trad.equals(q))
print("건수:", len(q))

In [ ]:
# 스스로 해보자! (4)
# 아래 빈칸(___)을 채우고 실행해보세요.

# result = (
#     orders
#     .___(subset=["amount"])
#     .query("0 < customer_age < 120")
#     .assign(
#         channel_clean=lambda d: d["channel"].str.___().str.___(),
#         amount_log=lambda d: ___,
#     )
#     .sort_values("amount", ascending=False)
#     .reset_index(drop=True)
# )
# print(result.shape)
# result.head(3)

## `.pipe()` — 함수 단위의 파이프라인 조립

# 5. `.pipe()` — 함수 단위의 파이프라인 조립

Part 4의 chaining은 **이 데이터**에 한 번 적용할 때는 깔끔합니다. 하지만 같은 정제를 다음 달 데이터에도, 또 다른 팀의 데이터셋에도 적용하려면 어떻게 해야 할까요?

매번 같은 chaining을 복사·붙여넣기? 그건 **다음 달의 여러분이 화낼 일** 입니다. 정제 단계를 **함수**로 만들고, 그 함수들을 chaining에 끼워 넣으면 됩니다.

> ❓ **이 파트에서 답할 질문:** 정제 단계 하나하나를 함수로 빼고, 한 줄에서 그 함수들을 차례로 조립하려면?

## 💡 쉽게 말하면 — chaining 안에서 호출되는 "내가 만든 메서드"

pandas의 `.pipe(함수)`는 다음과 같이 작동합니다.

```text
df.pipe(func)   ↔   func(df)
df.pipe(func, a, b)   ↔   func(df, a, b)
```

겉보기에 `df.head()`처럼 `.`을 찍는 형태이지만, 안쪽에서는 그냥 함수를 부르는 것일 뿐입니다. **내가 만든 함수가 마치 DataFrame의 메서드가 된 것처럼 chaining에 자연스럽게 끼어듭니다.**

## 자세히 알아보기

```python
def clean_region(df):           # 1번 입력 = DataFrame, return도 DataFrame
    return df.assign(
        region_clean=df["region"].str.strip().replace({"Seoul": "서울"})
    )

def filter_age(df, low=0, high=120):
    return df.query("@low < customer_age < @high")

result = (
    orders
    .pipe(clean_region)          # clean_region(orders)와 같음
    .pipe(filter_age, low=10, high=80)   # filter_age(.., low=10, high=80)와 같음
)
```

함수의 약속: **첫 입력은 DataFrame, 반환도 DataFrame**. 이 한 줄 약속만 지키면 어떤 함수든 `.pipe`로 끼울 수 있습니다.

> 💡 **개념 연결:** D+002 함수 편(`def calc_total(price, quantity)`)에서 함수가 "재사용 가능한 작업"이었죠. 여기선 그 함수의 *입력이 DataFrame*이고 *출력도 DataFrame*이라는 좁은 약속 하나가 추가된 것 뿐입니다. 그 약속만으로 chaining에 자연스럽게 들어옵니다.

> 📌 **실무 포인트:** 한 함수 = 한 단계 = 한 책임. 함수 이름이 곧 단계 설명이 되도록 짓습니다(`clean_region`, `filter_age`, `add_amount_log`). 그래야 파이프라인 한 줄만 봐도 무엇을 하는지 보입니다.

> **읽는 법:** 한 줄만 봐도 "문자열 정제 → 비정상값 제거 → 파생 컬럼 추가"라는 의도가 그대로 드러납니다. 같은 흐름을 다음 달의 새 데이터(`orders_next_month`)에도 적용하려면 **`orders` 자리만 바꾸면 됩니다**. 이게 재사용입니다.

## 인코딩과 스케일링까지 함수로

이제 Part 2·3에서 배운 인코딩과 스케일링도 함수로 빼서 끼워 넣어봅시다.

> **읽는 법:** 코드 한 *줄(엄밀히는 한 블록)* 이 "정제 → 결측·이상치 제거 → 파생 → 인코딩 → 스케일링"의 전체 흐름을 보여줍니다. **각 단계의 책임이 함수 이름에 다 들어 있어** 댓글이 거의 필요 없습니다.

## 데이터로 확인해 봅시다

- 재사용: 다음 달 데이터에도, 다른 팀의 비슷한 데이터에도, `pipeline_result = orders_next.pipe(...).pipe(...)`로 끝.
- 테스트: 각 함수 단위로 작은 가짜 데이터로 동작을 검증할 수 있습니다.
- 협업: 팀원이 코드를 읽을 때 함수 이름만 보고도 흐름을 이해합니다. (오늘의 핵심 태도)

> 📌 **다른 산업에서는?** 어디서나 같습니다. **금융**은 위험 점수 산출 파이프라인을 일·주·월 단위로 재실행하고, **마케팅**은 캠페인별 데이터에 같은 정제를 돌려 매번 보고서를 냅니다. **제조**는 설비별 센서 데이터에 같은 이상 탐지 파이프라인을, **헬스케어**는 병원별 임상 데이터에 같은 전처리를 적용해 모델을 학습합니다. 모두 "한 번 잘 만든 파이프라인을 여러 데이터에 반복 적용"하는 구조입니다.

> 💡 **더 알고 싶다면 (선택):** scikit-learn에는 `Pipeline`이라는 더 본격적인 도구가 있습니다. `Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])`처럼 *전처리부터 모델까지* 한 객체로 묶고, `fit`·`predict`도 그 객체 하나로 호출합니다. 오늘 익힌 `pipe`가 그 사고방식의 출발점입니다.

### 스스로 해보자! ✏️ (5)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

1. **(따라하기)** 위 `clean_strings`, `drop_invalid`, `add_features` 세 함수만 이용해 `orders_v2`를 만들어보세요. (인코딩·스케일링은 생략)
2. **(응용)** 새 함수 `add_amount_class(df)`를 만들어 `amount >= 100000`이면 `'high'`, 그 외 `'low'`인 `amount_class` 컬럼을 추가하세요. 그 함수를 파이프라인 마지막에 끼우세요.
3. **(생각해보기)** 같은 흐름을 함수가 아닌 chaining만으로 작성하면 어떤 점이 불편해질까요? (힌트: 다음 달 데이터에 똑같이 돌리려면?)

<details>
<summary>💡 힌트 (클릭)</summary>

```python
def add_amount_class(df):
    return df.assign(
        amount_class=np.where(df["amount"] >= 100_000, "high", "low")
    )

orders_v2 = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid)
    .pipe(add_features)
    .pipe(add_amount_class)
)
print(orders_v2[["amount", "amount_class"]].head())
print(orders_v2["amount_class"].value_counts())
```

함수만 추가하면 파이프라인에 끼우는 일은 한 줄(`.pipe(add_amount_class)`)이면 끝납니다. 책임이 분리돼 있어, 새 단계가 들어와도 기존 단계의 코드는 손대지 않습니다.

</details>

### ✅ 짚고 넘어가기

1. `df.pipe(func)`와 `func(df)`의 결과는 같습니다. 그런데도 `pipe`를 쓰는 이유는 무엇인가요?
2. 파이프에 끼울 함수가 지켜야 할 약속은 무엇인가요? (입력·출력)
3. 같은 흐름을 다음 달 데이터에 다시 적용해야 할 때, chaining-only와 `.pipe(함수)` 중 어느 쪽이 유리한가요?

> 💡 **다음 단계 예고:** 도구를 다 익혔으니 이제 진짜 일을 합니다. **CSV → 정제 → 인코딩/스케일링 → 저장**으로 이어지는 end-to-end 파이프라인을 종합 실습으로 완성합니다.

# 🧪 종합 실습 — CSV → 정제 → 저장 end-to-end 파이프라인

오늘의 도구를 모두 모아 **한 함수**로 묶어봅니다. 이 함수는 다음 달의 여러분(혹은 다른 팀원)이 새 데이터를 받았을 때 **한 줄로 부를** 수 있어야 합니다.

작업 흐름:

```text
[1] 원본 CSV 로드   →  [2] 정제   →  [3] 인코딩/스케일링   →  [4] 저장
```

각 단계를 함수로 만들고, `.pipe()`로 조립합니다. 마지막에 결과를 CSV로 저장하고 다시 읽어 검증합니다.

> 💡 **데이터셋:** 모두마켓 이번 달 주문이 CSV 파일로 들어왔다고 가정합니다. 아래 셀이 그 CSV를 임시 폴더에 만들어줍니다(자기완결적 스냅샷).

## 시나리오 1 — 단계 함수 정의 (한 함수 = 한 책임)

각 단계를 작은 함수로 빼서, 함수 이름만 봐도 무엇을 하는지 보이게 만듭니다.

## 시나리오 2 — 한 줄로 조립

방금 만든 함수들을 `.pipe()`로 이어 붙여 **end-to-end 파이프라인**을 만듭니다.

> **읽는 법:** `preprocess(input_csv)` 한 줄에 오늘 배운 모든 도구(`map`, 인코딩, 스케일링, chaining, `pipe`)가 다 녹아 있습니다. 다음 달 데이터가 들어와도 `preprocess(new_csv_path)` 한 줄이면 됩니다.

## 시나리오 3 — 저장 + 재로드로 검증

전처리 결과를 CSV로 저장하고, 다시 읽어 모양이 보존되는지 확인합니다. **저장이 끝나야 파이프라인이 끝난 것**입니다.

## 📊 품질 리포트 함수 만들기 (제출물)

여러분이 만든 파이프라인이 **어떤 행을 얼마나 걸러냈는지** 한눈에 보여주는 **품질 리포트 함수**를 작성하세요. 데이터 분석가의 무기는 결과만이 아니라 *결정의 근거*임을 기억하세요.

리포트가 출력해야 할 것:

- 입력 행 수 / 출력 행 수 / 제거된 비율
- 컬럼별 결측 비율(상위 5개)
- 새로 만들어진 파생/인코딩 컬럼 수
- 자료형 분포 요약

> 💡 **이 함수가 곧 다음 모듈(D+009 종합 프로젝트)에서 다시 등장합니다.** 오늘 잘 만들어두면 다음 산출물에 그대로 재활용됩니다.

> **읽는 법:** 이 리포트 하나면, *내일의 여러분*이 *오늘 어떤 결정을 내렸는지* 다시 볼 수 있습니다. "왜 200건이 사라졌지?"라는 질문에 "나이 0~120 필터에서 N건, 수량 10 초과에서 M건…"이라고 답할 수 있도록, 단계마다 카운트를 더 정교하게 분리하면 더욱 좋습니다.

## 🚀 더 나아가기 (선택)

- 각 단계 함수에 **로그**(어떤 행이 얼마나 걸러졌는지 print)를 추가해보세요.
- `preprocess`에 `verbose=True` 옵션을 더해, 켜면 단계별 카운트가 출력되도록 만들어보세요.
- 같은 파이프라인을 **다른 시드로 새로 만든 데이터**에 적용해, 입력이 바뀌어도 같은 함수로 동작하는지 확인하세요.

# ✅ 최종 확인 퀴즈

배운 내용을 잠깐 확인해볼게요. 틀려도 괜찮습니다. "이런 걸 배웠지" 하고 떠올리는 용도예요.

### 개념 퀴즈

1. `apply`, `map`, 벡터화 중 가장 빠르지만 가장 유연성이 떨어지는 것은? `(a) apply  (b) map  (c) 벡터화`
2. 다음 중 **순서가 없는** 카테고리에 가장 안전한 인코딩은? `(a) Label  (b) Ordinal  (c) One-Hot`
3. 이상치가 있는 수치형 컬럼에 가장 안전한 스케일러는? `(a) MinMax  (b) Standard  (c) Robust`
4. `df.pipe(func)`와 동등한 표현은? `(a) func.pipe(df)  (b) func(df)  (c) df.func()`

### 코드 퀴즈

`orders`에서 다음 작업을 **한 줄짜리 chaining**으로 작성하세요.

- `amount` 결측 제거
- `region`의 공백 제거(`str.strip()`)
- `customer_age`가 10~80인 행만 남기기
- `amount_log = log1p(amount)` 컬럼 추가
- `amount_log`로 정렬 (큰 순)
- 상위 5개만 보기

> **읽는 법:** 한 줄(블록)이 "결측 제거 → 공백 정리 → 나이 필터 → 파생 → 정렬 → 상위 5개"라는 흐름을 그대로 보여줍니다. **흐름이 곧 코드, 코드가 곧 흐름** — 이게 오늘의 핵심 메시지입니다.

# 🎓 정리 & 다음 시간 예고

## 오늘 배운 것이 어떻게 이어졌나

```text
[Part 1] apply · map · 벡터화        값 하나하나에 같은 변환을 거는 세 방법
   ↓  (벡터화가 가능하면 벡터화)
[Part 2] 범주형 인코딩                글자 카테고리를 모델이 읽을 숫자로
   ↓  (순서 있으면 Ordinal, 없으면 One-Hot)
[Part 3] 수치 스케일링                자릿수 다른 숫자를 한 자에 놓기
   ↓  (이상치 의심이면 Robust)
[Part 4] method chaining              임시 변수 없이 한 줄로 흐르게
   ↓  (단계가 흐름으로 읽힘)
[Part 5] .pipe()                      내가 만든 함수를 chaining에 끼우기
   ↓  (재사용 가능한 파이프라인 탄생)
[종합 실습] CSV → 정제 → 저장 end-to-end
```

## 한 장 정리표

| 주제 | 핵심 한 줄 | 대표 코드 |
| --- | --- | --- |
| apply/map/벡터화 | 가능하면 벡터화 → map → apply 순 | `s * 1.1`, `s.map(dict)`, `s.apply(f)` |
| 범주형 인코딩 | 순서 있으면 Ordinal, 없으면 One-Hot | `s.map({...})`, `pd.get_dummies(s)` |
| 수치 스케일링 | 이상치 의심이면 Robust | `RobustScaler().fit_transform(X)` |
| method chaining | 단계를 한 줄로 흐르게 | `df.dropna().query(...).assign(...)` |
| `.pipe()` | 내 함수를 chaining에 끼우기 | `df.pipe(func, **kwargs)` |
| end-to-end | 한 함수 = 한 책임, 한 줄로 호출 | `preprocess(path)` |

## 진단·판단 → 다음 분석 도구 매핑

| 오늘의 결과 | 자연스러운 다음 분석 |
| --- | --- |
| pandas로 만든 파이프라인이 **느리다**·메모리가 부족하다 | 청크 처리·dtype 최적화·**Polars** — 바로 다음 시간 |
| 파이프라인 결과를 **눈으로 검증**하고 싶다 | 시각화 심화(missingno·Seaborn·Plotly) — 며칠 뒤 |
| 정제된 데이터를 **모델에 넣고** 싶다 | scikit-learn의 `Pipeline`, 모델 학습으로 — 다음 모듈 |
| 같은 정제를 **다른 도메인**에 적용 | 함수 인자(`age_min`, `qty_max`)로 일반화 |

## 🎓 다음 시간 예고

> **"이 파이프라인이 대용량에서도 버티게 한다."**
>
> 오늘 만든 `preprocess()`는 1,500건에서는 잘 돌아갑니다. 하지만 *수십만 건의 웹 로그*(예: 주말 트래픽 50만 행)에서도 그럴까요? 다음 시간(D+007)에는 pandas의 메모리 한계, dtype 최적화, 청크 처리, 그리고 **Polars**의 lazy evaluation까지 — 대용량 데이터를 다루는 도구를 만납니다. 같은 파이프라인이 데이터가 커져도 *그대로* 또는 *살짝 바꾸어* 통하는지 확인합니다.

# 📝 오늘의 과제

오늘 만든 **end-to-end 파이프라인**과 **품질 리포트 함수**를 다듬어 GitHub에 제출합니다.

## 제출물

1. 오늘의 종합 실습 노트북(`.ipynb`) — 파이프라인 + 품질 리포트 포함
2. 저장된 `orders_clean.csv`(또는 동등한 산출물)

## 필수 과제

- [ ] 단계 함수 5개 이상을 정의했고, 각 함수가 **한 책임**만 진다.
- [ ] `preprocess()` 함수 한 줄 호출로 전체 파이프라인이 동작한다.
- [ ] 인코딩(Ordinal + One-Hot)과 스케일링(Robust 등) 모두 포함한다.
- [ ] 결과를 CSV로 저장하고, **다시 읽어** shape가 유지되는 것까지 확인했다.
- [ ] 품질 리포트 함수가 입력·출력 행 수, 제거 비율, 결측 상위 5개를 보고한다.

## 심화 과제 (선택)

- [ ] `verbose=True` 옵션으로 각 단계의 카운트를 출력하게 만들었다.
- [ ] 같은 파이프라인을 **새 데이터**(시드 변경 또는 다른 csv)로 검증했고, 결과 차이를 글로 적었다.
- [ ] 인코딩 전략을 바꾼 **두 버전**(예: One-Hot vs Target Encoding 흉내)을 비교했다.

## 제출 방법 (GitHub)

개인 포트폴리오 저장소에 `D006/` 폴더를 만들어 노트북을 올립니다.

```bash
# 2주차 이후로 PR 흐름을 익혀갑니다.
git checkout -b feature/d006-pipeline
git add D006/
git commit -m "D006 전처리 파이프라인 제출"
git push origin feature/d006-pipeline
# 그리고 GitHub에서 main으로 PR을 엽니다.
```

## 평가 기준

| 축 | 무엇을 보나 |
| --- | --- |
| 정확성 | 단계 함수와 파이프라인이 오류 없이 돌고, 결과가 일관된가 |
| 합리성 | 인코딩·스케일링 전략 선택이 변수 성격에 맞는가 |
| 재사용성 | 함수 이름·인자·문서화가 *다음 사람*에게 친절한가 |
| 인사이트 | 품질 리포트의 수치와 그 의미를 글로 남겼는가 |

> 💡 모두의연구소의 과제는 순위를 매기지 않습니다. **"어제의 나보다 더 깔끔하고 재사용 가능한 코드를 썼는가"** 가 기준이에요. 같은 함수를 다른 사람도 한 줄로 부를 수 있다면 성공입니다.

---

수고하셨습니다! 🎉

오늘 여러분은 데이터 분석가의 두 번째 큰 무기, **"반복을 구조화하는 힘"** 을 손에 넣었습니다. 같은 정제를 매번 손으로 하지 않고, 한 함수로 묶어 한 줄로 부를 수 있게 됐어요. 이제 다음 달의 여러분이 한결 가벼울 거예요.

다음 시간에는 이 파이프라인이 **대용량에서도** 버티게 만드는 법을 배웁니다. 천천히, 그러나 멈추지 않고 가봅시다. 🚀

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 교안은 생성형 AI를 활용해 제작하고 제작자가 검수했습니다.<br>
무단 복제 및 배포를 금합니다.</sub>

In [ ]:
# 예제: 단계마다 함수로 빼기 — '한 함수 = 한 책임'
def clean_strings(df):
    # 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid(df, age_min=0, age_max=120, qty_max=20):
    # 불가능한 값(나이 범위·과대 수량)과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
    )

def add_features(df):
    # 분석에 쓸 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
    )

print("세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.")

In [ ]:
# 예제: 한 줄로 조립
cleaned = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
)

print("원본:", orders.shape, "→ 정제 후:", cleaned.shape)
cleaned.head(3)

In [ ]:
# 예제: 인코딩/스케일링 함수 추가
from sklearn.preprocessing import RobustScaler

def encode_categories(df):
    # membership은 Ordinal, region·channel·category는 One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    # One-Hot
    out = pd.concat(
        [out,
         pd.get_dummies(out["region"], prefix="region", dtype=int),
         pd.get_dummies(out["channel"], prefix="ch", dtype=int),
         pd.get_dummies(out["category"], prefix="cat", dtype=int)],
        axis=1
    )
    return out

def scale_numeric(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 RobustScaler로 스케일링.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in cols], index=df.index)
    return pd.concat([df, scaled_df], axis=1)

# 전체 파이프라인 한 줄
pipeline_result = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
    .pipe(encode_categories)
    .pipe(scale_numeric)
)

print("최종 shape:", pipeline_result.shape)
print("새로 생긴 컬럼 일부:", [c for c in pipeline_result.columns if "scaled" in c or c.startswith("region_")][:8])

In [ ]:
# 스스로 해보자! (5)
# def add_amount_class(df):
#     return df.assign(
#         amount_class=np.where(df["amount"] >= 100_000, "high", "low")
#     )

# orders_v2 = (
#     orders
#     .pipe(clean_strings)
#     .pipe(drop_invalid)
#     .pipe(add_features)
#     .pipe(add_amount_class)   # 새 함수 끼우기
# )
# print(orders_v2[["amount", "amount_class"]].head())
# print(orders_v2["amount_class"].value_counts())

In [ ]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비 (실무에서는 외부에서 받아오는 단계)
# ─────────────────────────────────────────────
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

In [ ]:
# 시나리오 1 — 단계 함수들
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)

def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")

In [ ]:
# 시나리오 2 — end-to-end 파이프라인
def preprocess(input_path):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )

# 진짜 한 줄 호출
clean_df = preprocess(input_csv)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

In [ ]:
# 시나리오 3 — 저장 + 재로드 검증
clean_df.to_csv(output_csv, index=False)

reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)

In [ ]:
# 품질 리포트 함수
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    # 전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.
    report = {
        "rows_before": before.shape[0],
        "rows_after":  after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after":  after.shape[1],
        "new_cols":    after.shape[1] - before.shape[1],
        "missing_top5_before": (
            before.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "missing_top5_after": (
            after.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }
    return report

# 사용
rep = quality_report(orders, clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

In [ ]:
# 코드 퀴즈 — 모범 답안
top5 = (
    orders
    .dropna(subset=["amount"])
    .assign(region=lambda d: d["region"].str.strip())
    .query("10 < customer_age < 80")
    .assign(amount_log=lambda d: np.log1p(d["amount"]))
    .sort_values("amount_log", ascending=False)
    .head(5)
)
print(top5[["region", "customer_age", "amount", "amount_log"]])